In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('groupproject_2/vk_eda/articles_match_status.csv')
df.head()

,article,status,brand,name,rating,reviews,n_mentions,total_likes,total_views,first_mention,last_mention
0,211466804,matched,wetty_lab,Крем для тела увлажняющий питательный для ухода,4.8,229.0,22,5,464,2024-12-18 15:29:51,2026-04-06 18:53:42
1,414386668,matched,art&fact,"Энзимная пудра для умывания с витамином C, 65 г",4.8,9933.0,9,360,8060,2025-07-29 10:35:59,2026-01-29 12:35:48
2,414386668,matched,art&fact,"Энзимная пудра для умывания с витамином C, 65 г",4.8,9935.0,9,360,8060,2025-07-29 10:35:59,2026-01-29 12:35:48
3,139625239,matched,vois,Пенка для умывания лица увлажняющая очищающая ...,4.8,86492.0,5,18,4693,2024-06-24 12:00:02,2025-04-27 07:40:00
4,139625239,matched,vois,Пенка для умывания лица увлажняющая очищающая ...,4.8,86492.0,5,18,4693,2024-06-24 12:00:02,2025-04-27 07:40:00


In [3]:
df[df['status'] == 'only_vk']

,article,status,brand,name,rating,reviews,n_mentions,total_likes,total_views,first_mention,last_mention
189,523231435,only_vk,NaN,NaN,NaN,NaN,6,383,11018,2025-10-30 08:54:26,2025-12-21 17:00:02
190,688613778,only_vk,NaN,NaN,NaN,NaN,3,118,6164,2025-01-22 15:36:34,2025-09-16 14:27:03
191,701890634,only_vk,NaN,NaN,NaN,NaN,3,30,4606,2025-12-17 08:40:16,2025-12-23 14:40:19
192,309272983,only_vk,NaN,NaN,NaN,NaN,3,227,18565,2025-01-31 14:08:12,2025-04-08 14:12:05
193,246538773,only_vk,NaN,NaN,NaN,NaN,2,113,7338,2024-12-12 16:00:48,2025-01-10 16:16:36
194,230900707,only_vk,NaN,NaN,NaN,NaN,2,11,2908,2024-11-12 09:00:01,2025-04-21 08:40:09
195,521169963,only_vk,NaN,NaN,NaN,NaN,2,76,1201,2025-12-08 11:10:00,2025-12-30 15:40:35
196,218602044,only_vk,NaN,NaN,NaN,NaN,2,73,3682,2025-05-27 09:30:17,2025-07-17 10:44:32
197,148822805,only_vk,NaN,NaN,NaN,NaN,2,0,1705,2024-11-15 15:30:00,2024-12-17 09:00:00
198,265293502,only_vk,NaN,NaN,NaN,NaN,2,12,1565,2024-10-14 08:51:22,2024-10-14 09:07:08


In [4]:
missed_article = set(df[df['status'] == 'only_vk']['article'].unique())

Сначала проверим, что отстутсвующие артикли пропали не из-за неправильного приведения брендов

In [5]:
df_first = pd.read_csv('groupproject_2/parsed/wb_data.csv')
df_first.head()

,id,brand,name,price,old_price,discount,rating,reviews,link,search_url
0,695213222,ART&FACT.,Шампунь для жирных волос очищающий 400 мл,502,9462.0,NaN,"4,8",1264.0,https://www.wildberries.ru/catalog/695213222/d...,https://www.wildberries.ru/catalog/0/search.as...
1,791357100,ART&FACT.,"Сыворотка для лица с витамином С, набор 1+1",680,1710.0,NaN,"4,7",17219.0,https://www.wildberries.ru/catalog/791357100/d...,https://www.wildberries.ru/catalog/0/search.as...
2,44151357,ART&FACT.,"Лосьон для лица от акне и комедонов, 150 мл",525,2890.0,NaN,"4,7",10425.0,https://www.wildberries.ru/catalog/44151357/de...,https://www.wildberries.ru/catalog/0/search.as...
3,158573601,ART&FACT.,Пенка для умывания от прыщей,384,11020.0,NaN,"4,8",23762.0,https://www.wildberries.ru/catalog/158573601/d...,https://www.wildberries.ru/catalog/0/search.as...
4,694507304,ART&FACT.,"Шампунь для волос от выпадения, 400 мл",411,7753.0,NaN,"4,8",1264.0,https://www.wildberries.ru/catalog/694507304/d...,https://www.wildberries.ru/catalog/0/search.as...


In [6]:
len(missed_article)

47

In [7]:
len(set(df_first['id'].unique()) & missed_article)

0

Нет пересечений, поэтому спрасим только требуемые артикулы

In [8]:
import time
import csv
import logging
import os
import re
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

csv_file = "groupproject_2/parsed/wb_missed_articles.csv"
log_file = "groupproject_2/logs/wb_missed.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.FileHandler(log_file, encoding="utf-8"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger()

def clean_number(text):
    if not text:
        return ""
    return re.sub(r"[^\d.]", "", text).strip()

def load_existing_cards(csv_file):
    existing = set()
    if not os.path.exists(csv_file):
        return existing
    with open(csv_file, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            article = (row.get("article") or "").strip()
            if article:
                existing.add(article)
    return existing

def open_csv_for_append(csv_file):
    file_exists = os.path.exists(csv_file)
    csv_f = open(csv_file, "a", encoding="utf-8", newline="")
    fieldnames = ["article", "brand", "name", "price", "old_price", "rating", "reviews", "url"]
    writer = csv.DictWriter(csv_f, fieldnames=fieldnames)
    if not file_exists or os.path.getsize(csv_file) == 0:
        writer.writeheader()
    return csv_f, writer

def get_product_data(driver, article):
    url = f"https://www.wildberries.ru/catalog/{article}/detail.aspx"
    data = {
        "article": str(article),
        "brand": "",
        "name": "",
        "price": "",
        "old_price": "",
        "rating": "",
        "reviews": "",
        "url": url,
    }
    try:
        driver.get(url)
        WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
        driver.execute_script("window.scrollTo(0, 500)")
        time.sleep(3)

        try:
            name_el = driver.find_element(By.CSS_SELECTOR, "[class*='productTitle']")
            data["name"] = name_el.text.strip()
        except:
            try:
                name_el = driver.find_element(By.CSS_SELECTOR, "h1")
                data["name"] = name_el.text.strip()
            except:
                pass

        try:
            brand_el = driver.find_element(By.CSS_SELECTOR, "[class*='brandBadgeText']")
            data["brand"] = brand_el.text.strip()
        except:
            pass

        try:
            price_el = driver.find_element(By.CSS_SELECTOR, "[class*='priceBlockFinalPrice']")
            data["price"] = clean_number(price_el.text)
        except:
            try:
                price_el = driver.find_element(By.CSS_SELECTOR, "[class*='finalPrice']")
                data["price"] = clean_number(price_el.text)
            except:
                pass

        try:
            old_el = driver.find_element(By.CSS_SELECTOR, "[class*='priceBlockOldPrice']")
            data["old_price"] = clean_number(old_el.text)
        except:
            pass

        try:
            rating_el = driver.find_element(By.CSS_SELECTOR, "[class*='ratingNumberMinimized']")
            data["rating"] = rating_el.text.strip()
        except:
            try:
                rating_el = driver.find_element(By.CSS_SELECTOR, "[class*='ratingValue']")
                data["rating"] = rating_el.text.strip()
            except:
                pass

        try:
            rev_el = driver.find_element(By.CSS_SELECTOR, "[class*='feedbackCountMinimized']")
            data["reviews"] = clean_number(rev_el.text)
        except:
            try:
                rev_el = driver.find_element(By.CSS_SELECTOR, "[class*='reviewCount']")
                data["reviews"] = clean_number(rev_el.text)
            except:
                pass

    except Exception as e:
        logger.warning(f"Ошибка {article}: {e}")
    return data

options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)

existing = load_existing_cards(csv_file)
csv_f, writer = open_csv_for_append(csv_file)

logger.info(f"Всего артикулов: {len(missed_article)}")
total_new = 0

for article in missed_article:
    art_str = str(article).strip()
    if not art_str or art_str in existing:
        logger.info(f"Пропускаем {art_str}")
        continue

    logger.info(f"Парсим {art_str}")
    data = get_product_data(driver, art_str)
    writer.writerow(data)
    csv_f.flush()
    total_new += 1
    existing.add(art_str)
    logger.info(f"Сохранён {art_str}")

logger.info(f"Добавлено: {total_new}")
csv_f.close()
driver.quit()

2026-04-10 19:55:34 - INFO - Всего артикулов: 47
2026-04-10 19:55:34 - INFO - Парсим 306061825
2026-04-10 19:55:40 - INFO - Сохранён 306061825
2026-04-10 19:55:40 - INFO - Парсим 142821506
2026-04-10 19:55:44 - INFO - Сохранён 142821506
2026-04-10 19:55:44 - INFO - Парсим 309017858
2026-04-10 19:55:49 - INFO - Сохранён 309017858
2026-04-10 19:55:49 - INFO - Парсим 309017861
2026-04-10 19:55:53 - INFO - Сохранён 309017861
2026-04-10 19:55:53 - INFO - Парсим 469666182
2026-04-10 19:55:59 - INFO - Сохранён 469666182
2026-04-10 19:55:59 - INFO - Парсим 311659017
2026-04-10 19:56:08 - INFO - Сохранён 311659017
2026-04-10 19:56:08 - INFO - Парсим 549253521
2026-04-10 19:56:12 - INFO - Сохранён 549253521
2026-04-10 19:56:12 - INFO - Парсим 688613778
2026-04-10 19:56:17 - INFO - Сохранён 688613778
2026-04-10 19:56:17 - INFO - Парсим 246162580
2026-04-10 19:56:23 - INFO - Сохранён 246162580
2026-04-10 19:56:23 - INFO - Парсим 246538773
2026-04-10 19:56:26 - INFO - Сохранён 246538773
2026-04-10 

Избавимся от артикулов, которых нет на вб

In [25]:
wb = pd.read_csv('groupproject_2/parsed/wb_missed_articles.csv')


In [26]:
wb = wb[wb['name'] != 'По Вашему запросу ничего не найдено'].copy()
wb.shape

(39, 8)

In [27]:
wb.info()

<class 'pandas.DataFrame'>
Index: 39 entries, 0 to 46
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   article    39 non-null     int64  
 1   brand      0 non-null      float64
 2   name       38 non-null     str    
 3   price      17 non-null     float64
 4   old_price  18 non-null     float64
 5   rating     0 non-null      float64
 6   reviews    0 non-null      float64
 7   url        39 non-null     str    
dtypes: float64(5), int64(1), str(2)
memory usage: 2.7 KB


Посмотрим на данные и избавимся от товаров, не связанных с косметикой

In [28]:
article_delete = [309017858, 309017861, 549253521, 688613778, 309272983, 309549871, 309582013, 309103821, 953399007, 772700896, 309104495, 380321650 ]
wb = wb[~wb['article'].isin(article_delete)].copy()
wb.shape

(27, 8)

Соединим их с уже имеющимися данными с вб

In [29]:
wb_before = pd.read_csv('groupproject_2/wb_eda.csv')
wb_before.columns

Index(['id', 'brand', 'name', 'price', 'old_price', 'rating', 'reviews',
       'discount'],
      dtype='str')

In [30]:
wb.columns

Index(['article', 'brand', 'name', 'price', 'old_price', 'rating', 'reviews',
       'url'],
      dtype='str')

In [31]:
wb_before.rename(columns={'id': 'article'}, inplace=True)
wb.drop(columns=['url'], inplace=True)
wb['discount'] = (wb['old_price'] - wb['price']) / wb['old_price'] * 100

In [32]:
print(wb.columns)
print(wb_before.columns)

Index(['article', 'brand', 'name', 'price', 'old_price', 'rating', 'reviews',
       'discount'],
      dtype='str')
Index(['article', 'brand', 'name', 'price', 'old_price', 'rating', 'reviews',
       'discount'],
      dtype='str')


In [33]:
wb_all = pd.concat([wb, wb_before])
wb_all.columns

Index(['article', 'brand', 'name', 'price', 'old_price', 'rating', 'reviews',
       'discount'],
      dtype='str')

In [34]:
data = pd.read_csv('groupproject_2/vk_eda/all_posts_clean.csv')
data = data.dropna(subset=['article'])
data = data[data['views_count'] >= data['likes_count']]
data['article'] = data['article'].astype('int')

data.head()

,brand,date,text,likes_count,comments_count,reposts_count,views_count,article_or_link,final_url,article,тип
1,artfact.products,2026-04-02 14:38:36,"Первое апреля прошло, а, значит, это точно не ...",24,2,0,1025,https://clc.to/SpQOHQ,https://www.wildberries.ru/catalog/884634860/d...,884634860,ссылка на WB
2,artfact.products,2026-03-31 09:00:21,Когда продукт становится не просто баночкой в ...,40,3,0,1000,https://vk.cc/cW35g2,https://www.wildberries.ru/catalog/760104825/d...,760104825,ссылка на WB
3,artfact.products,2026-03-26 12:28:16,Да кто такие эти ваши микроиглы? Без паники — ...,53,4,0,1014,https://vk.cc/cVU14O,https://www.wildberries.ru/catalog/807378841/d...,807378841,ссылка на WB
4,artfact.products,2026-03-24 09:00:09,У тебя тоже бывает так: стоишь перед зеркалом ...,34,0,1,793,https://vk.cc/cVOzit,https://www.wildberries.ru/catalog/695213222/d...,695213222,ссылка на WB
5,artfact.products,2026-03-23 08:48:52,Слышишь? Так звучит твой новый ASMR-ритуал 🤫🎧\...,33,0,2,440,https://vk.cc/cVMDzr,https://www.wildberries.ru/catalog/695213222/d...,695213222,ссылка на WB


In [43]:
data['brand'].unique()

<StringArray>
['artfact.products',  'geltekcosmetics',         'iconskin',
           'aravia',        'theu_club',             'vois',
            'mixit',        'wetty_lab']
Length: 8, dtype: str

In [44]:
data['brand'] = 'art&fact' if 'artfact.products' in data['brand'] else data['brand']

In [45]:
all_df = pd.merge(data, wb_all, on='article', how='right')

In [46]:
all_df.columns

Index(['brand_x', 'date', 'text', 'likes_count', 'comments_count',
       'reposts_count', 'views_count', 'article_or_link', 'final_url',
       'article', 'тип', 'brand_y', 'name', 'price', 'old_price', 'rating',
       'reviews', 'discount'],
      dtype='str')

In [48]:
all_df.loc[all_df['brand_y'].isna(), 'brand_y'] = all_df.loc[all_df['brand_y'].isna(), 'brand_x']

all_df.drop(columns=['brand_x'], inplace = True)
all_df.rename(columns={'brand_y': 'brand'}, inplace = True)
all_df.columns

Index(['date', 'text', 'likes_count', 'comments_count', 'reposts_count',
       'views_count', 'article_or_link', 'final_url', 'article', 'тип',
       'brand', 'name', 'price', 'old_price', 'rating', 'reviews', 'discount'],
      dtype='str')

In [49]:
all_df.head()

,date,text,likes_count,comments_count,reposts_count,views_count,article_or_link,final_url,article,тип,brand,name,price,old_price,rating,reviews,discount
0,2025-04-14 12:06:59,"Мужчины тоже устают, и им требуется перерыв на...",72.0,1.0,0.0,2649.0,306061825,NaN,306061825,артикул,artfact.products,NaN,NaN,2444.0,NaN,NaN,NaN
1,2023-02-02 14:54:56,Особенная НОВИНКА 🤍 \r\n \r\nВ дерматологическ...,134.0,3.0,2.0,2532.0,https://vk.cc/clgCaM,https://www.wildberries.ru/catalog/142821506/d...,142821506,ссылка на WB,iconskin,Увлажняющий лосьон для чувствительной кожи лиц...,NaN,NaN,NaN,NaN,NaN
2,2025-08-12 12:49:42,"Продукт, который подарит коже красивый оттенок...",31.0,0.0,0.0,2732.0,https://vk.cc/cOwlZB,https://www.wildberries.ru/catalog/469666182/d...,469666182,ссылка на WB,geltekcosmetics,Капли автозагар для лица и тела концентрат,NaN,NaN,NaN,NaN,NaN
3,2025-02-25 09:20:07,Новинка: гидрофильный бальзам для умывания и с...,50.0,6.0,2.0,3081.0,https://vk.cc/cIVwyb,https://www.wildberries.ru/catalog/311659017/d...,311659017,ссылка на WB,geltekcosmetics,Гидрофильный бальзам щербет для умывания и сня...,1411.0,2098.0,NaN,NaN,32.745472
4,2024-12-19 16:04:55,Бесконечно можно смотреть на три вещи: как гор...,43.0,0.0,0.0,2679.0,246162580,NaN,246162580,артикул,artfact.products,Подарочный набор масок для лица,NaN,NaN,NaN,NaN,NaN


In [52]:
all_df.to_csv('groupproject_2/analysis/wb_eda.csv', index=False)